# Entropy in Optimal Transport, and Sinkhorn

Notes and demos from the **Computational Optimal Transport** reading seminar (Summer 2026).

Main reference: Peyré & Cuturi, *Computational Optimal Transport* ([arXiv:1803.00567](https://arxiv.org/abs/1803.00567)), especially the entropic OT / Sinkhorn chapter.

---

## Recap (from my notes)

### Monge → Kantorovich

**Monge.** Find a map $T:X\to Y$ that pushes $\mu$ to $\nu$ as cheaply as possible:

$$\min_T\left\{\int_X c\!\left(x,T(x)\right)\,d\mu(x):\; T_{\#}\mu=\nu\right\},\qquad (T_{\#}\mu)(A)=\mu\!\left(T^{-1}(A)\right).$$

Mass is conserved, but mass **cannot split**: each source point goes to exactly one target.

**Kantorovich.** Relax to a coupling $\gamma$ (a joint plan $P$) on $X\times Y$:

$$\min_{\gamma\in\Pi(\mu,\nu)}\int_{X\times Y} c\,d\gamma.$$

Now mass **can split**. Discrete version: $P$ is a non-negative matrix with prescribed margins.

### Exact OT

$$L_C(a,b)=\min_{P\in U(a,b)}\langle C,P\rangle=\min_{P\in U(a,b)}\sum_{ij}C_{ij}P_{ij}$$

- $L_C$ — cheapest shipping cost  
- $P$ — transport plan  
- $C$ — cost table  
- $U(a,b)$ — feasible set: non-negative matrices with row sums $a$ and column sums $b$

Goal: ship supply $a$ (source $\sim\mu$) to demand $b$ (target $\sim\nu$) at minimum cost.

### Entropic OT

Add $-\varepsilon H(P)$. The flat polytope objective becomes a **smooth bowl**; the optimum slides off the corner into the interior as $\varepsilon\uparrow$. Strictly convex $\Rightarrow$ unique $P^{\varepsilon}$.

$$L_C^{\varepsilon}(a,b)=\min_{P\in U(a,b)}\langle C,P\rangle-\varepsilon H(P)$$

**Sinkhorn.** Solve via dual scalings $u,v$: alternate $u=a/(Kv)$, $v=b/(K^{\top}u)$ — fix rows to $a$, fix cols to $b$, repeat (the oscillation) until both lock.

$$\text{stationarity}\;\Longrightarrow\; P^{\varepsilon}=\operatorname{diag}(u)\,K\,\operatorname{diag}(v),\quad K=e^{-C/\varepsilon}.$$

### What this notebook covers

0. Setup  
1. Build the pieces $(a,b,C,K)$  
2. Sinkhorn from scratch  
3. Check against POT  
4. Demos ($\varepsilon$-sweep, bounce, 2D, speed)  
5. Takeaways


## 0. Setup

Imports and [POT](https://pythonot.github.io/) (Python Optimal Transport).

Naming map (notes ↔ POT): cost matrix `M` is our $C$, and `reg` is our $\varepsilon$.


In [ ]:
!pip install pot -q

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import ot, time

%matplotlib inline
plt.rcParams.update({"figure.dpi":110, "font.size":11, "axes.grid":False})
COUP = "viridis"
print("POT version:", ot.__version__)

## 1. Build the pieces ($a,\ b,\ C,\ K$)

Two structured (bimodal) distributions on a line, squared-distance cost $C_{ij}=(x_i-y_j)^2$, and the Gibbs kernel $K=e^{-C/\varepsilon}$.

- $C$ = how far / how expensive  
- $K$ = the same table flipped into desirability: cheap routes glow bright, dear routes go dark


In [ ]:
def normalize(v):
    v = np.asarray(v, float); return v / v.sum()

def mixture(x, mus, sigmas, ws):
    g = sum(w*np.exp(-(x-mu)**2/(2*s*s)) for mu, s, w in zip(mus, sigmas, ws))
    return g / g.sum()

n = m = 16
x = np.linspace(0, 1, n)
y = np.linspace(0, 1, m)
C = (x[:, None] - y[None, :])**2                    # cost table
a = mixture(x, [0.28, 0.70], [0.07, 0.06], [0.6, 0.4])   # bimodal source
b = mixture(y, [0.42, 0.86], [0.09, 0.05], [0.5, 0.5])   # bimodal target
eps = 0.02
K = np.exp(-C / eps)
print("a sums to", round(a.sum(),6), "| b sums to", round(b.sum(),6), "| shapes", C.shape)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8.2, 3.4))
im0 = ax[0].imshow(C, origin="lower", cmap="magma_r"); ax[0].set_title("cost  C  (dark = cheap)")
im1 = ax[1].imshow(K, origin="lower", cmap=COUP);      ax[1].set_title(r"kernel  $K=e^{-C/\varepsilon}$  (bright = desirable)")
for a_ in ax: a_.set_xlabel("target j"); a_.set_ylabel("source i")
fig.colorbar(im0, ax=ax[0], fraction=.046); fig.colorbar(im1, ax=ax[1], fraction=.046)
plt.tight_layout(); plt.show()

## 2. Sinkhorn from scratch

From the notes: form $K=e^{-C/\varepsilon}$, then iterate

$$u \leftarrow \frac{a}{Kv},\qquad v \leftarrow \frac{b}{K^{\top}u}.$$

Intuition: squeeze the rows so each carries its supply $a$; squeeze the columns so each carries its demand $b$; repeat. Rebuild the plan as

$$P=\operatorname{diag}(u)\,K\,\operatorname{diag}(v).$$

This is matrix scaling (Sinkhorn–Knopp) on a strictly positive kernel — convergence is guaranteed (**Sinkhorn's theorem**).


In [ ]:
def sinkhorn(a, b, C, eps, n_iter=1000, log=False):
    K = np.exp(-C / eps)
    u = np.ones_like(a)
    v = np.ones_like(b)
    history = []
    for it in range(n_iter):
        u = a / (K @ v)                 # fix rows:  row sums -> a
        v = b / (K.T @ u)               # fix cols:  col sums -> b
        if log:
            P = u[:, None] * K * v[None, :]
            history.append((P.sum(1).copy(), P.sum(0).copy(), P.copy()))
    P = u[:, None] * K * v[None, :]     # P = diag(u) K diag(v)
    return (P, history) if log else P

In [ ]:
P = sinkhorn(a, b, C, eps)
print("row sum error |P1 - a|   =", np.abs(P.sum(1) - a).max())
print("col sum error |P^T 1 - b| =", np.abs(P.sum(0) - b).max())
print("transport cost <C,P>      =", (P * C).sum())

## 3. Sinkhorn (from scratch) vs POT

Our plan should match `ot.sinkhorn`, and both should approach the exact LP plan `ot.emd` as $\varepsilon\to 0$.

**The trade.** $\langle C,P^{\varepsilon}\rangle \ge \langle C,P^{\star}\rangle = L_C(a,b)$: the blurred (entropic) plan always costs at least as much as the sharp exact plan — never less. Entropy buys smoothness and speed; you pay a little extra transport cost.


In [ ]:
P_ours  = sinkhorn(a, b, C, eps)
P_pot   = ot.sinkhorn(a, b, C, reg=eps)
P_exact = ot.emd(a, b, C)

print("max |ours - POT| :", np.abs(P_ours - P_pot).max())
print("cost ours / POT  :", round((P_ours*C).sum(),6), "/", round((P_pot*C).sum(),6))
print("cost exact (P*)  :", round((P_exact*C).sum(),6), "  (entropic cost is always >= this)")

## 4. Demos

Visual checks of the story in the notes: blur vs sharpness, the row/column oscillation, and why Sinkhorn scales better than a combinatorial LP.

Interactive ε blur (separate page when published): [ε slider](../assets/epsilon_slider.html) — on the live site: [epsilon_slider.html](./epsilon_slider.html).


### $\varepsilon$ sweep: blurry → sharp (the entropy story)

Two bimodal histograms. Shrink $\varepsilon$ and watch $P^{\varepsilon}$ tighten from a smeared cloud toward the sharp exact plan, while the transport cost drops toward the exact cost.

Smaller $\varepsilon$ $\to$ sharper plan, lower cost. Entropic cost is always $\ge$ exact, and $\to$ exact as $\varepsilon\to 0$.


In [ ]:
N = 64
xs = np.linspace(0, 1, N)
aS = mixture(xs, [0.22, 0.58], [0.05, 0.06], [0.6, 0.4])
bS = mixture(xs, [0.40, 0.82], [0.07, 0.05], [0.5, 0.5])
CS = (xs[:, None] - xs[None, :])**2
PexS = ot.emd(aS, bS, CS)

epss = [1.0, 0.1, 0.02, 0.005]
fig, ax = plt.subplots(1, len(epss)+1, figsize=(3*(len(epss)+1), 3))
for k, e in enumerate(epss):
    Pe = sinkhorn(aS, bS, CS, e)
    ax[k].imshow(Pe, origin="lower", cmap=COUP)
    ax[k].set_title(f"$\\varepsilon$={e}\ncost={(Pe*CS).sum():.3f}"); ax[k].set_xticks([]); ax[k].set_yticks([])
ax[-1].imshow(PexS, origin="lower", cmap=COUP)
ax[-1].set_title(f"exact $P^*$\ncost={(PexS*CS).sum():.3f}"); ax[-1].set_xticks([]); ax[-1].set_yticks([])
plt.suptitle("smaller $\\varepsilon$  $\\rightarrow$  sharper plan, lower cost", y=1.05); plt.tight_layout(); plt.show()

In [ ]:
es = np.geomspace(2.0, 0.003, 25)
costs = [ (sinkhorn(aS, bS, CS, e)*CS).sum() for e in es ]
plt.figure(figsize=(6,3.2))
plt.semilogx(es, costs, "-o", ms=4, label=r"entropic cost $\langle C,P^\varepsilon\rangle$")
plt.axhline((PexS*CS).sum(), ls="--", color="crimson", label="exact cost")
plt.gca().invert_xaxis(); plt.xlabel(r"$\varepsilon$ (shrinking $\rightarrow$)"); plt.ylabel("transport cost")
plt.legend(); plt.title(r"entropic cost $\geq$ exact, $\rightarrow$ exact as $\varepsilon\rightarrow 0$"); plt.tight_layout(); plt.show()

### 1D bimodal coupling (bell drawing)

Source $a$ and target $b$ each have two bumps (cf. textbook Fig. 4.2 in Peyré & Cuturi). The plan $P$ fills the square; its margins stay pinned to $a$ (bottom) and $b$ (left) while only the interior blurs with $\varepsilon$.


In [ ]:
N = 100; t = np.linspace(0, 1, N)
aG = mixture(t, [0.25, 0.55], [0.05, 0.05], [0.6, 0.4])
bG = mixture(t, [0.45, 0.80], [0.06, 0.05], [0.45, 0.55])
CG = (t[:, None] - t[None, :])**2
PG = sinkhorn(aG, bG, CG, eps=0.0015)

fig = plt.figure(figsize=(5.6, 5.6))
gs  = fig.add_gridspec(2, 2, width_ratios=[1,4], height_ratios=[4,1], wspace=0.04, hspace=0.04)
axM = fig.add_subplot(gs[0,1]); axL = fig.add_subplot(gs[0,0], sharey=axM); axB = fig.add_subplot(gs[1,1], sharex=axM)
axM.imshow(PG.T, origin="lower", extent=[0,1,0,1], cmap=COUP, aspect="auto")
axM.set_xticks([]); axM.set_yticks([]); axM.set_title("the coupling $P^\\varepsilon$")
axB.plot(t, aG, color="seagreen"); axB.fill_between(t, aG, color="seagreen", alpha=.3)
axB.set_xlabel("source a (green)"); axB.set_yticks([])
axL.plot(bG, t, color="crimson"); axL.fill_betweenx(t, bG, color="crimson", alpha=.3)
axL.invert_xaxis(); axL.set_ylabel("target b (red)"); axL.set_xticks([])
plt.show()

### Watch it bounce (the oscillation)

Start from $K$, then alternate **one** squeeze at a time.

- **Fix rows:** row sums lock onto $a$ ($\checkmark$), columns drift.  
- **Fix cols:** column sums lock onto $b$ ($\checkmark$), rows drift.

The drift shrinks every full round until both margins lock together — an oscillation that dies out.

Guaranteed because $K=e^{-C/\varepsilon}>0$ entrywise: **Sinkhorn's theorem**.


In [ ]:
# the bounce: one squeeze per frame, exactly like the interactive stepper in the notes
nb = 6
xb = np.linspace(0, 1, nb)
Cb = (xb[:, None] - xb[None, :])**2
ab = normalize([0.05, 0.26, 0.10, 0.30, 0.17, 0.12])
bb = normalize([0.22, 0.08, 0.28, 0.12, 0.10, 0.20])
Kb = np.exp(-Cb / 0.04)

frames = [("start", Kb.copy())]
M = Kb.copy()
for _ in range(9):
    M = M * (ab / M.sum(1))[:, None]; frames.append(("row", M.copy()))   # fix rows
    M = M * (bb / M.sum(0))[None, :]; frames.append(("col", M.copy()))   # fix cols

fig, ax = plt.subplots(figsize=(6.4, 6.4))
def bounce(k):
    ax.clear()
    action, Mk = frames[k]
    rs, cs = Mk.sum(1), Mk.sum(0); mx = Mk.max()
    ax.imshow(Mk, origin="upper", cmap=COUP, extent=[0, nb, nb, 0], vmin=0, vmax=mx)
    for i in range(nb):
        for j in range(nb):
            ax.text(j+0.5, i+0.5, f"{Mk[i,j]:.2f}", ha="center", va="center",
                    fontsize=7, color=("white" if Mk[i,j] > mx*0.5 else "black"))
    for i in range(nb):
        ok = abs(rs[i]-ab[i]) < 2e-3
        ax.text(nb+0.12, i+0.5, (f"{rs[i]:.2f} \u2713" if ok else f"{rs[i]:.2f} \u2192{ab[i]:.2f}"),
                va="center", ha="left", fontsize=9, family="monospace", color=("seagreen" if ok else "0.55"))
    for j in range(nb):
        ok = abs(cs[j]-bb[j]) < 2e-3
        ax.text(j+0.5, nb+0.12, (f"{cs[j]:.2f}\n\u2713" if ok else f"{cs[j]:.2f}\n\u2192{bb[j]:.2f}"),
                va="top", ha="center", fontsize=8, family="monospace", color=("seagreen" if ok else "0.55"))
    ax.text(nb+0.12, -0.25, "row \u03a3", fontsize=9, family="monospace", color="0.4")
    ax.text(-0.1, nb+0.12, "col \u03a3", fontsize=9, family="monospace", color="0.4", va="top", ha="right")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlim(-0.3, nb+1.7); ax.set_ylim(nb+1.3, -0.5)
    title = {"start":"start: this is K  (neither margin locked)",
             "row":"fix ROWS  \u2192  rows locked \u2713   (columns drift)",
             "col":"fix COLS  \u2192  columns locked \u2713   (rows drift)"}[action]
    err = max(np.abs(rs-ab).max(), np.abs(cs-bb).max())
    ax.set_title(f"{title}\nmax margin error = {err:.4f}", fontsize=11)
anim = animation.FuncAnimation(fig, bounce, frames=len(frames), interval=750)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# the oscillation as one curve: margin error shrinking each full round
_, hist = sinkhorn(ab, bb, Cb, 0.04, n_iter=15, log=True)
errs = [max(np.abs(r-ab).max(), np.abs(c-bb).max()) for (r, c, _) in hist]
plt.figure(figsize=(6,3.2))
plt.semilogy(range(1, len(errs)+1), errs, "-o", ms=4)
plt.xlabel("full round"); plt.ylabel("max margin error (log)")
plt.title("the oscillation dies out geometrically"); plt.tight_layout(); plt.show()

### Exact vs entropic, side by side

$P^{\star}$ (exact) sits on corners of the polytope — sharp. $P^{\varepsilon}$ lives in the interior — blurry. The blur costs a little:

$$\langle C,P^{\varepsilon}\rangle \ge \langle C,P^{\star}\rangle.$$

That gap is the price of entropy (and of a unique, differentiable solution).


In [ ]:
Pe = sinkhorn(a, b, C, 0.01)
fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
ax[0].imshow(P_exact, origin="lower", cmap=COUP); ax[0].set_title(f"exact $P^*$  (cost {(P_exact*C).sum():.4f})")
ax[1].imshow(Pe, origin="lower", cmap=COUP);      ax[1].set_title(f"entropic $P^\\varepsilon$  (cost {(Pe*C).sum():.4f})")
for a_ in ax: a_.set_xlabel("target j"); a_.set_ylabel("source i")
gap = (Pe*C).sum() - (P_exact*C).sum()
plt.suptitle(f"price of blur = extra cost = {gap:.4f}  (always >= 0)", y=1.04); plt.tight_layout(); plt.show()

### 2D transport: a ring to two clusters

A more realistic point-cloud setting (uniform weights, squared Euclidean cost). Sinkhorn returns a soft coupling; we draw a line for each significant $P_{ij}$ (thicker = more mass).


In [ ]:
rng = np.random.default_rng(3)
th  = np.linspace(0, 2*np.pi, 70, endpoint=False)
src = np.column_stack([np.cos(th), np.sin(th)]) * 1.3 + rng.normal(0, 0.05, (70, 2))
c1  = rng.normal([3.4,  1.2], 0.28, (35, 2))
c2  = rng.normal([3.4, -1.2], 0.28, (35, 2))
tgt = np.vstack([c1, c2])
wa = np.ones(len(src))/len(src); wb = np.ones(len(tgt))/len(tgt)
C2 = ((src[:, None, :] - tgt[None, :, :])**2).sum(-1)
P2 = sinkhorn(wa, wb, C2, eps=0.06)

plt.figure(figsize=(7.2, 4.2))
thr = P2.max()*0.12
for i in range(len(src)):
    for j in range(len(tgt)):
        if P2[i, j] > thr:
            plt.plot([src[i,0], tgt[j,0]], [src[i,1], tgt[j,1]],
                     color="gray", lw=7*P2[i,j]/P2.max(), alpha=0.3, zorder=1)
plt.scatter(*src.T, c="seagreen", s=26, zorder=2, label="source (ring)")
plt.scatter(*tgt.T, c="crimson",  s=26, zorder=2, label="target (2 clusters)")
plt.legend(loc="upper left"); plt.title("entropic transport: a ring splits to two clusters")
plt.gca().set_aspect("equal"); plt.tight_layout(); plt.show()

### Speed: exact vs Sinkhorn

Exact OT (`ot.emd`) is a combinatorial LP, classically roughly $O(n^3)$ in the worst case. Sinkhorn is $O(nm)$ matrix–vector work per iteration.

**Note.** On CPU at small/medium $n$, wall times can look comparable (NumPy BLAS vs POT's fast C++ simplex). Sinkhorn's real advantages show up with GPU parallelism, very large $n$, and **differentiability** — none of which exact OT offers cleanly.


In [ ]:
sizes = [100, 200, 400, 700, 1000, 1400]
t_emd, t_sink = [], []
for s in sizes:
    xx = np.linspace(0, 1, s); CC = (xx[:,None]-xx[None,:])**2
    av = np.ones(s)/s; bv = np.ones(s)/s
    t0=time.time(); ot.emd(av, bv, CC);              t_emd.append(time.time()-t0)
    t0=time.time(); sinkhorn(av, bv, CC, 0.01, 10); t_sink.append(time.time()-t0)
plt.figure(figsize=(6,3.4))
plt.plot(sizes, t_emd,  "-o", label="exact  ot.emd")
plt.plot(sizes, t_sink, "-o", label="Sinkhorn (10 iter)")
plt.xlabel("n (problem size)"); plt.ylabel("runtime (s)")
plt.legend(); plt.title("exact OT bends up faster as n grows"); plt.tight_layout(); plt.show()

## 5. Takeaways

**Primal $\to$ dual.** We want $P\in\mathbb{R}^{n\times m}$ ($n\cdot m$ numbers). We actually solve for dual scalings $u=e^{f/\varepsilon}$, $v=e^{g/\varepsilon}$ ($n+m$ numbers), via $P=\operatorname{diag}(u)\,K\,\operatorname{diag}(v)$. When $n=m$: $n^2$ unknowns vs $2n$.

**The algorithm is two divisions.** $u=a/(Kv)$, $v=b/(K^{\top}u)$, repeat. Each step fixes one margin; the other drifts a shrinking amount (the oscillation) until both lock. Sinkhorn's theorem guarantees this because $K>0$.

**The $\varepsilon$ trade.** Small $\varepsilon$ $\to$ sharp, near-exact, numerically stiffer. Large $\varepsilon$ $\to$ fast, smooth, blurrier. Entropic cost is always $\ge$ exact.

**Why bother.** Speed at scale (parallel / GPU) and a smooth gradient (usable as a trainable loss).

**In a nutshell.** Know the pickup fee $f$ at each source and the dropoff fee $g$ at each target, and the whole route table $P$ is determined.

---

## References

1. Gabriel Peyré & Marco Cuturi, *Computational Optimal Transport*, Foundations and Trends in Machine Learning, 2019. [arXiv:1803.00567](https://arxiv.org/abs/1803.00567)  
2. Marco Cuturi, *Sinkhorn Distances: Lightspeed Computation of Optimal Transport*, NeurIPS 2013.  
3. [Python Optimal Transport (POT)](https://pythonot.github.io/) library used for `ot.sinkhorn` / `ot.emd` checks.

Handwritten notes from the seminar live in this repo under `notes/`.
